# 01 — EDA: Startup Funding Dataset
**Goal:** Understand the data before modeling. Every chart answers a question a VC analyst asks manually.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/sample/startups_sample.csv')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Quick data quality check
print('Missing values:')
print(df.isnull().sum())
print(f'\nSectors: {df["sector"].nunique()} unique')
print(f'Stages:  {df["stage"].nunique()} unique')
print(f'Countries: {df["hq_country"].nunique()} unique')

In [ ]:
# Sector distribution — this is the classification target
sector_counts = df['sector'].value_counts()

plt.figure(figsize=(12, 5))
sector_counts.plot(kind='barh', color=sns.color_palette('husl', len(sector_counts)))
plt.title('Sector Distribution (Classification Target)', fontweight='bold')
plt.xlabel('Count')
plt.tight_layout()
plt.savefig('../data/sample/eda_sectors.png', dpi=120)
plt.show()

print(f'Imbalance ratio: {sector_counts.max()/sector_counts.min():.1f}x  →  use class_weight=balanced')

In [ ]:
# Funding stage distribution
stage_order = ['Seed', 'Series A', 'Series B', 'Series C']
stage_counts = df['stage'].value_counts().reindex(stage_order).fillna(0)

plt.figure(figsize=(8, 4))
stage_counts.plot(kind='bar', color=['#2ecc71','#3498db','#9b59b6','#e74c3c'], edgecolor='white')
plt.title('Funding Stage Distribution', fontweight='bold')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../data/sample/eda_stages.png', dpi=120)
plt.show()

In [ ]:
# Funding amount by sector
df['funding_M'] = df['funding_amount_usd'] / 1e6
sector_funding = df.groupby('sector')['funding_M'].median().sort_values()

plt.figure(figsize=(12, 5))
sector_funding.plot(kind='barh', color='steelblue')
plt.title('Median Funding ($M) by Sector', fontweight='bold')
plt.xlabel('Median Funding ($M)')
plt.tight_layout()
plt.savefig('../data/sample/eda_funding.png', dpi=120)
plt.show()

In [ ]:
# Text length — drives tokenizer and TF-IDF decisions
df['word_count'] = df['description'].str.split().str.len()

plt.figure(figsize=(8, 4))
df['word_count'].hist(bins=15, color='steelblue', edgecolor='white')
plt.axvline(df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {df["word_count"].mean():.0f}')
plt.title('Description Word Count Distribution', fontweight='bold')
plt.xlabel('Word Count')
plt.legend()
plt.tight_layout()
plt.savefig('../data/sample/eda_text_length.png', dpi=120)
plt.show()

print(df['word_count'].describe().round(1))
print(f'\nTokenizer max_length recommendation: {int(df["word_count"].quantile(0.95))*2} tokens')

In [ ]:
# Key findings
print('=== KEY FINDINGS ===')
print(f'Classes         : {df["sector"].nunique()} sectors')
print(f'Imbalance       : {sector_counts.max()/sector_counts.min():.1f}x  →  class_weight=balanced')
print(f'Avg description : {df["word_count"].mean():.0f} words')
print(f'TF-IDF features : 5000 recommended')
print(f'Train/test split: 80/20 stratified by sector')
print(f'Largest deal    : ${df["funding_M"].max():.0f}M — {df.loc[df["funding_M"].idxmax(),"name"]}')